In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
# install relevant material
!pip install transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00


### PDF-Datei hochladen
Bitte laden Sie Ihre PDF-Datei hoch. Nach dem Hochladen ist sie im `/content/` Verzeichnis verfügbar.

In [3]:
from google.colab import files

# upload material 
uploaded = files.upload()

# checking uploaded material
print("Hochgeladene Dateien:")
for Steuerbuch2026_DE_BF in uploaded.keys():
  print(f'- {Steuerbuch2026_DE_BF}')

Saving Steuerbuch2026_DE_BF.pdf to Steuerbuch2026_DE_BF.pdf
Hochgeladene Dateien:
- Steuerbuch2026_DE_BF.pdf


In [4]:
# library allows Python to read and extract text from PDF files
!pip install pypdf

from pypdf import PdfReader
# create empty string variable to store all extracted text
reader = PdfReader("Steuerbuch2026_DE_BF.pdf")

# create empty string for text
text = ""
# loop through each page of the PDF document and append to text
for page in reader.pages:
    text += page.extract_text() + "\n"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 6.3 MB/s eta 0:00:00


In [5]:
# checking text
print(text[:200])

Das Steuerbuch 
2026
Tipps zur Arbeitnehmer ­
veranlagung 2025 für  
Lohnsteuerzahler/innen

Das Steuerbuch 2026
Tipps zur Arbeitnehmerveranlagung 2025 
für Lohnsteuerzahler/innen
Wien 2025
Impressum



In [7]:
# a specific extraction method for this Steuerbuch generated by ChatGPT
import re

lines = text.split("\n")
qa_pairs = []
current_question = None
current_answer_lines = []

# regex to detect page numbers/table of contents markers at the end of a line
toc_or_rz_pattern = re.compile(r"(\.{2,}\s*\d+|\s*Rz\s+\d+\s*f+)", re.IGNORECASE)

# function to clean a question line from Rz references or trailing page numbers
def clean_question(q_text):
    q_text = q_text.strip()
    # remove Rz references or similar short notes at the end of a question
    q_text = re.sub(r'\s*Rz\s+\d+\s*f+\s*$', '', q_text, flags=re.IGNORECASE)
    # remove trailing dots and numbers (like page numbers in TOC entries)
    q_text = re.sub(r'[\.\s]+\d+\s*$', '', q_text)
    return q_text.strip()

for line in lines:
    clean_line = line.strip()

    # skip very short or empty lines, or lines that are clearly just page numbers or roman numeral headers
    if not clean_line or len(clean_line) < 5 or re.fullmatch(r"^\d+\s*$", clean_line) or re.fullmatch(r"^[IVXLCDM]+\.", clean_line):
        continue

    # check if the line contains a question mark and is not a table of contents entry
    is_question_candidate = "?" in clean_line and not toc_or_rz_pattern.search(clean_line)

    # further filter out lines that look like headings but happen to have a question mark
    # heuristics: if it starts with Roman numeral or Capital letter followed by a dot, and then text, it might be a heading.
    is_heading_like = re.match(r"^[IVXLCDM]+\.\s+|^[A-Z]\.\s+", clean_line) and "?" in clean_line

    if is_question_candidate and not is_heading_like:
        # new potential question is found, save the previous QA pair if valid
        if current_question and current_answer_lines and len(" ".join(current_answer_lines).strip()) > 0:
            qa_pairs.append({
                "question": clean_question(current_question),
                "answer": " ".join(current_answer_lines).strip()
            })

        current_question = clean_line
        current_answer_lines = []
    elif current_question:
        # accumulate answer lines for the current question
        current_answer_lines.append(clean_line)

# add the last QA pair if it exists
if current_question and current_answer_lines and len(" ".join(current_answer_lines).strip()) > 0:
    qa_pairs.append({
        "question": clean_question(current_question),
        "answer": " ".join(current_answer_lines).strip()
    })

# check pairs
print("Total QA pairs:", len(qa_pairs))

Total QA pairs: 136


In [8]:
# retrieve relevant QA pairs
clean_qa = []

for qa in qa_pairs:
    q = qa["question"]
    a = qa["answer"]

    # filter 1: question must be meaningful (not too short)
    if len(q) < 15:
        continue

    # filter 2: question must contain '?'
    if "?" not in q:
        continue

    # filter 3: answer should not be empty or too short if it's the only content
    if len(a.strip()) < 10: # Reduced from 50, now checking for minimal content
        continue

    # add the QA pair if it passes all checks
    clean_qa.append({
        "question": q,
        "answer": a
    })

print(f"Number of clean QA pairs: {len(clean_qa)}")

Number of clean QA pairs: 113


In [9]:
formatted_data = []

for qa in clean_qa:
    formatted_data.append({
        "text": f"""### Instruction:
{qa['question']}

### Response:
{qa['answer']}"""
    })

In [10]:
# import Python's built-in JSON module
# module allows you to read and write data in JSON format
import json
import json

with open("train_data.json", "w", encoding="utf-8") as f:
    json.dump(formatted_data, f, ensure_ascii=False, indent=2)

In [11]:
# check questions
for i in range(5):
    print(formatted_data[i]["text"])
    print("-"*50)

### Instruction:
Worin unterscheiden sich Lohn- und Einkommensteuer?

### Response:
Grundsätzlich gilt: Arbeitnehmer/innen sowie Pensionistinnen /  Pensionisten zahlen Lohnsteuer, Selbständige zahlen Einkommensteuer. Die Lohnsteuer unterscheidet sich von der Einkommensteuer lediglich in ihrer Erhebungsform. Der Steuertarif ist grundsätzlich gleich. Für Arbeitnehmer/innen gibt es aber zusätzliche Absetzbeträge, besondere Steuerbefreiungen und Sonderbestim ­ mungen für die Besteuerung bestimmter „ sonstiger Bezüge“. Die Lohnsteuer hat jeder Arbeitgeber einzubehalten und bis zum 15. des Folgemonats an das Finanzamt abzuführen.  Rz 1194–1202a Die Einkommensteuer wird im Veranlagungsweg erhoben. Dazu ist eine Einkommensteuererklärung beim Finanzamt abzugeben. Auf Grund dieser Erklärung wird die Einkommensteuer ermittelt und mit Einkommensteuerbe ­ scheid vorgeschrieben. Bei der Veranlagung werden auch die nichtselbstän ­ digen Einkünfte miteinbezogen. Die von der Lohnverrechnung bereits ein

In [12]:
# check length
lengths = [len(qa["answer"]) for qa in clean_qa]

print("Max length:", max(lengths))
print("Min length:", min(lengths))

Max length: 19197
Min length: 36


In [13]:
# define a function that shortens text to a maximum length (default 2000 characters)
# helps avoid overly long training samples that could slow training or exceed token limits
def truncate(text, max_length=2000):
    return text[:max_length]

clean_qa = [
    {"question": qa["question"], "answer": truncate(qa["answer"])}
    for qa in clean_qa
]

In [14]:
# define a function that checks whether an answer is suitable for training
def is_good_answer(a):
     # keep only answers that
    # -are longer than 50 characters
    # -don't start with "r" (likely filtering incomplete or unwanted responses)
    return len(a) > 50 and not a.lower().startswith("r")

# filter dataset
clean_qa = [qa for qa in clean_qa if is_good_answer(qa["answer"])]

In [15]:
# install libraries needed for transformer-based model training
# - transformers: hugging Face models
# - datasets: dataset handling
# - peft: parameter-efficient fine-tuning (LoRA)
# - accelerate: speeds up GPU training
# - bitsandbytes: enables 4-bit quantization for memory efficiency
!pip install -q transformers datasets peft accelerate bitsandbytes

In [16]:
import torch # (deep learning framework)
from datasets import load_dataset # import dataset loader from Hugging Face
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer # import tokenizer and causal language model tools
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training # import LoRA configuration tools for parameter-efficient fine-tuning

In [17]:
# import BitsAndBytes configuration class
# used for memory-efficient 4-bit model loading
from transformers import BitsAndBytesConfig

In [18]:
# define configuration for 4-bit quantization
# this reduces GPU memory usage while keeping model performance high
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # load model weights in 4-bit precision instead of 16/32-bit
    bnb_4bit_compute_dtype=torch.float16,  # float16 for internal computations (efficient)
    bnb_4bit_use_double_quant=True, # enables double quantization for compression accuracy
    bnb_4bit_quant_type="nf4" # (nf4) quantization type (recommended for LLM training)
)

In [19]:
# import tokenizer and causal language model again
from transformers import AutoModelForCausalLM, AutoTokenizer
# mistral-7b-instruct-v0.2 (instruction-tuned large language model)
model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# load the tokenizer for the selected model
tokenizer = AutoTokenizer.from_pretrained(model_name) # Initialize tokenizer
# set the padding token equal to the end-of-sequence token
tokenizer.pad_token = tokenizer.eos_token

# load the pretrained causal language model
# causal language models predict the next token step-by-step (used for text generation)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,  # applies 4-bit quantization to reduce memory usage
    device_map="auto"   # automatically distributes the model across available gpu/cpu devices
)
# set the model padding token id equal to the tokenizer eos token id
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [20]:
!pip install -q bitsandbytes accelerate transformers peft

In [21]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# prepare model for efficient low-bit (4-bit) training
model = prepare_model_for_kbit_training(model)


# define lora configuration (parameter-efficient fine-tuning setup)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# apply lora adapters to the model
model = get_peft_model(model, lora_config)

In [22]:
# check parameters
model.print_trainable_parameters()

trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [23]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="train_data.json")

Generating train split: 0 examples [00:00, ? examples/s]

In [24]:
tokenizer.pad_token = tokenizer.eos_token

In [25]:
def tokenize(example):
    # convert text into tokens the model can read
    tokenized_inputs = tokenizer(
        example["text"],        # input text field from dataset
        truncation=True,        # cut text if longer than max_length
        padding="max_length",   # pad shorter sequences to fixed length
        max_length=512          # set maximum sequence length
    )
    # set labels equal to input_ids for causal language model training
    tokenized_inputs["labels"] = tokenized_inputs["input_ids"].copy()
    return tokenized_inputs
# apply tokenization function to entire dataset
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/113 [00:00<?, ? examples/s]

### Training des Modells
Nun, da die Daten vorbereitet und das Modell konfiguri ist, können wir den eigentlichen Trainingsprozess starten. Wir werden `TrainingArguments` definieren, um die Hyperparameter für das Training festzulegen, und dann den `Trainer` von Hugging Face nutzen, um das Modell mit den tokenisierten Daten zu trainieren.

In [26]:
from transformers import TrainingArguments, Trainer

# define training configuration
training_arguments = TrainingArguments(
    output_dir="./results",              # folder where checkpoints and logs are saved
    num_train_epochs=3,                  # number of full training passes over dataset
    per_device_train_batch_size=4,       
    gradient_accumulation_steps=1,       
    optim="paged_adamw_32bit",           # memory-efficient optimizer for quantized training
    save_steps=100,                      # save model checkpoint every 100 steps
    logging_steps=100,                   # log training metrics every 100 steps
    learning_rate=2e-4,                  # step size for parameter updates
    weight_decay=0.001,                  
    fp16=False,                          # disable float16 precision training
    bf16=False,                          # disable bfloat16 precision training
    max_grad_norm=0.3,                   
    max_steps=-1,                        
    warmup_ratio=0.03,                   # gradual learning rate warmup at start of training
    group_by_length=True,                # group similar-length samples for efficiency
    lr_scheduler_type="constant",        
    report_to="tensorboard"              
)

# initialize trainer with model and dataset
trainer = Trainer(
    model=model,                                  # pretrained model with lora adapters
    train_dataset=tokenized_dataset["train"],      # training dataset
    eval_dataset=tokenized_dataset["train"],       # evaluation dataset (same as train here)
    args=training_arguments                       # training configuration
)

# start model fine-tuning
trainer.train()

# save final fine-tuned model to disk
trainer.save_model("./fine_tuned_mistral")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


In [ ]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_pro

In [29]:
import re  # import regex library for sentence detection

def generate_answer(question):
    # create instruction-style prompt for mistral instruct model
    prompt = f"""### Instruction:
{question}

### Response:
"""

    # tokenize prompt and move tensors to same device as model
    inputs = tokenizer(
        prompt,
        return_tensors="pt",   
        truncation=True,       # cut input if too long
        max_length=512         # max input length
    ).to(model.device)

    # generate model response
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,              # limit response length
        do_sample=True,                  # enable sampling (less deterministic)
        temperature=0.5,                 # lower randomness, more precise answers
        top_p=0.9,                       # sampling for controlled diversity
        pad_token_id=tokenizer.eos_token_id  # avoid padding warnings
    )

    # convert tokens back into readable text
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # extract only the response part after "### response:"
    if "### Response:" in decoded:
        decoded = decoded.split("### Response:")[-1].strip()

    # find sentence endings
    sentence_enders = list(re.finditer(r'[.!?]', decoded))

    if sentence_enders:
        # cut text at last full sentence ending
        last_ender = sentence_enders[-1]
        decoded = decoded[:last_ender.end()]
    else:
        # fallback: shorten text if no sentence ending found
        decoded = decoded[:100]

    return decoded  # return cleaned final answer

In [30]:
# text
print(generate_answer("Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?"))

Die Bemessungsgrundlage für die Körperschaftsteuer ist der Lohn ­  und Gehaltsbetrag der Arbeitnehmer, die für die Verrechnung zuständig sind, sowie die sonstigen Einkünfte der Körperschaft, die nach der Einkommensteuer ­ richtlinie zu versteuern sind.


In [32]:
from google.colab import files
uploaded = files.upload()

Saving dataset_clean.csv to dataset_clean.csv


In [33]:
# check questions 
import pandas as pd

df = pd.read_csv("dataset_clean.csv")

df.head()

,id,prompt
0,CORP-TAX-001,Was ist die steuerliche Bemessungsgrundlage fü...
1,CORP-TAX-002,"Welche steuerlichen Konsequenzen hat es, wenn ..."
2,CORP-TAX-003,"Welche Körperschaften sind verpflichtet, sämtl..."
3,CORP-TAX-004,Wie wird eine Dividende einer österreichischen...
4,CORP-TAX-005,Was unterscheidet die steuerliche Behandlung e...


In [34]:
answers = []  # create empty list to store generated answers

for i, row in df.iterrows():  # loop through each row in dataframe
    question = row["prompt"]  # extract question text from prompt column

    try:
        answer = generate_answer(question)  # generate model answer
    except Exception as e:
        print(f"Error at row {i}: {e}")  # print error if generation fails
        answer = "ERROR"  # store placeholder if error occurs

    answers.append(answer)  # save answer to list

    if i % 50 == 0:  # print progress every 50 rows
        print(f"{i}/{len(df)} done")  # show processing status

0/643 done
Error at row 22: name 'max_new_tokens' is not defined
Error at row 49: name 'max_new_tokens' is not defined
Error at row 50: name 'max_new_tokens' is not defined
50/643 done
Error at row 61: name 'max_new_tokens' is not defined
100/643 done
Error at row 101: name 'max_new_tokens' is not defined
150/643 done
200/643 done
Error at row 238: name 'max_new_tokens' is not defined
250/643 done
Error at row 295: name 'max_new_tokens' is not defined
300/643 done
350/643 done
400/643 done
Error at row 418: name 'max_new_tokens' is not defined
Error at row 419: name 'max_new_tokens' is not defined
Error at row 422: name 'max_new_tokens' is not defined
450/643 done
500/643 done
550/643 done
Error at row 574: name 'max_new_tokens' is not defined
Error at row 575: name 'max_new_tokens' is not defined
Error at row 598: name 'max_new_tokens' is not defined
600/643 done
Error at row 630: name 'max_new_tokens' is not defined
Error at row 642: name 'max_new_tokens' is not defined


In [35]:
df["answer"] = answers

In [36]:
submission = df[["id", "answer"]]

In [37]:
submission.to_csv("finetuneanswer.csv", index=False)

In [38]:
from google.colab import files
files.download("finetuneanswer.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>